In [1]:
%matplotlib widget
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import scipy
import h5py
import os
print(os.getpid())
%cd ../../

import pylib.mix as mix
import cvxpy as cp
import pylib.qucf_read as qucf_r
import pylib.measurement as mse

import pylib.Chebyschev_coefs as ch

from matplotlib import colors
colors_ = ["blue", "red", "green", "gray", "black"]

plt.rcParams.update({
    "text.usetex": True,
    'text.latex.preamble': r"\usepackage{amsmath} \boldmath"
})

22160
/media/work/docs/codes/QuCF/scripts-py


In [2]:
path_save_scan_ = "./jupyter-notebooks/Stepanoff/results/"
path_save_HS_ = "../QuCF/simulations/Stepanoff/Pauli/hs"
path_qucf_ = "../QuCF/simulations/Stepanoff/Pauli"
for i in range(100):
    plt.close()

import qiskit
from qiskit.quantum_info import Operator, SparsePauliOp
from qiskit.circuit.library import PauliEvolutionGate
from qiskit import QuantumCircuit
from qiskit.circuit.library import DiagonalGate
print("qiskit version: ", qiskit.version.get_version_info())

qiskit version:  1.3.2


In [3]:
# ---------------------------------------------------
# --- Helper functions ---
# ---------------------------------------------------
def plot_matrix(
        A, Nx, 
        name_A = "", label_A = "", 
        step_tick = 10,
        fontsize = 30, marker_size = 80,
        text_coord_label = [1, 14],
        text_coord_name_A = [12, 1],
):
    def find_rows_columns(A):
        N = A.shape[0]
        rows_plot_1 = np.zeros(N*N)
        cols_plot_1 = np.zeros(N*N)
        N_nz_1 = 0
        for ir in range(N):
            for ic in range(N):
                if np.abs(A[ir, ic]) > 0:
                    N_nz_1 += 1
                    rows_plot_1[N_nz_1-1] = ir
                    cols_plot_1[N_nz_1-1] = ic
        rows_plot_1 = rows_plot_1[0:N_nz_1]
        cols_plot_1 = cols_plot_1[0:N_nz_1]
        return rows_plot_1, cols_plot_1
    # -----------------------------------------------
    marker_size = int(marker_size / (Nx/4.)**2)

    N = A.shape[0]
    rows_A, cols_A = find_rows_columns(A)

    fig1 = plt.figure(figsize=(11,10))
    ax = fig1.add_subplot(111)
    ax.scatter(cols_A, rows_A, color="red", s = marker_size) 
    plt.xlim(0, N - 1)
    plt.ylim(0, N - 1)
    plt.gca().invert_yaxis()
    plt.xlabel(r'$\textbf{columns}$', fontsize = fontsize)
    plt.ylabel(r"$\textbf{rows}$", fontsize = fontsize)
    plt.grid()

    plt.yticks(np.arange(0, N, step_tick))
    plt.xticks(np.arange(0, N, step_tick))

    ax.tick_params(axis='both', which='major', labelsize=fontsize)

    if Nx > 4:
        # ax.set_xticks([])
        ax.tick_params(axis='both', which='major', labelbottom=False, labelleft=False)


    # ax.text(
    #     int(text_coord_label[0]), int(text_coord_label[1]), 
    #     r'$\textbf{' + label_A + '}$', fontsize=fontsize
    # )
    
    # ax.text(
    #     int(text_coord_name_A[0]), int(text_coord_name_A[1]), 
    #     r'$' + name_A + '$', fontsize=fontsize
    # )

    # --- Indicate blocks ---
    for ii in range(Nx):
        ax.hlines(y=ii * Nx - 0.5, xmin=0, xmax=N, linewidth=2, color='black')
        ax.vlines(x=ii * Nx - 0.5, ymin=0, ymax=N, linewidth=2, color='black')

    # --- plot diagonal ---
    for ii in range(0, Nx):
        ax.plot([ii*Nx,N], [0, N-ii*Nx], "--b")
    for ii in range(1, Nx):
        ax.plot([0,N-ii*Nx], [ii*Nx, N], "--b")

    plt.show()
    return
# ---------------------------------------------------------------------------------------
# plot_matrix(V_, Nx_, "M", "M", step_tick = 1)

In [4]:
# -------------------------------------------------------------------------------
# --- Create a diagonal matrix with x-coordinates on the main diagonal ---
# -------------------------------------------------------------------------------
def create_diag_matrix(n, alpha, name_matrix):
    N = 1 << n
    aa = np.linspace(0, 2.*np.pi, N)
    # diag = np.zeros(N**2, dtype=np.float64)
    diag = np.zeros(N**2)
    count = -1

    if name_matrix == "M1":
        for i2 in range(N):
            for i1 in range(N):
                count += 1
                diag[count] = (1. - alpha) * i1 * (1. - np.cos(aa[i2]))

    if name_matrix == "M2":
        for i2 in range(N):
            for i1 in range(N):
                count += 1
                diag[count] = alpha * i2 * (1. - np.cos(aa[i1]))

    A = np.diag(diag)
    op = Operator(A)
    A_dec = SparsePauliOp.from_operator(op)

    print("Number of Pauli products in the matrix {:s}: {:d}".format(name_matrix, len(A_dec)))

    # --- reference unitary matrix ---
    Nsq = N**2
    diag_U_ref = np.zeros(Nsq, dtype = complex)
    for ii in range(Nsq):
        diag_U_ref[ii] = np.exp(-1j * t_ * A[ii, ii])
    diag_U_ref = np.sort(diag_U_ref)

    return A, A_dec, name_matrix, diag_U_ref
# ---------------------------------------------------------------------------
def show_nonzero_Pauli_products(A_dec, name_A, flag_show):
    if flag_show:
        print("--- Pauli products for the matrix {:s} ---".format(name_A))
        for prod in A_dec:
            print(prod.paulis, prod.coeffs)
    return
# ---------------------------------------------------------------------------
def build_qiskit_circuit(A_dec, name_A, n, t, flag_circuit):
    # --- Plot the evolution circuit in Qiskit ---
    U_qiskit = None
    if flag_circuit:
        print("--- Circuit of Hamiltonian simulation for the propagator {:s} ---".format(name_A))

        # Create evolution gate
        evolution_gate = PauliEvolutionGate(A_dec, time = t)

        # Add it to a quantum circuit
        qc = QuantumCircuit(2*n)
        qc.append(evolution_gate, range(2*n))

        # Decompose into explicit gates
        decomposed_qc = qc.decompose()
        print(decomposed_qc)

        # # Get the unitary representing the circuit 
        # U_qiskit =  Operator(decomposed_qc).data
    return U_qiskit
# ---------------------------------------------------------------------------
flag_show_Pauli_products, flag_circuit = True, True

t_ = 1.0
n_ = 2
N_ = 1 << n_
alpha_ = np.sqrt(20.)

M1_, M1_dec_, name_M1_, diag_U_M1_ = create_diag_matrix(n = n_, alpha = alpha_, name_matrix="M1")
M2_, M2_dec_, name_M2_, diag_U_M2_ = create_diag_matrix(n = n_, alpha = alpha_, name_matrix="M2")

print("\n---------------------------------------")
show_nonzero_Pauli_products(M1_dec_, name_M1_, flag_show_Pauli_products)

print("\n---------------------------------------")
show_nonzero_Pauli_products(M2_dec_, name_M2_, flag_show_Pauli_products)

print("\n-------------------------------------------------------------------------")
U_M1_qiskit_ = build_qiskit_circuit(M1_dec_, name_M1_, n_, t_, flag_circuit)

print("\n-------------------------------------------------------------------------")
U_M2_qiskit_ = build_qiskit_circuit(M2_dec_, name_M2_, n_, t_, flag_circuit)

del flag_show_Pauli_products, flag_circuit

Number of Pauli products in the matrix M1: 6
Number of Pauli products in the matrix M2: 6

---------------------------------------
--- Pauli products for the matrix M1 ---
['IIII'] [-3.906+0.j]
['IIIZ'] [1.302+0.j]
['IIZI'] [2.604+0.j]
['ZZII'] [3.906+0.j]
['ZZIZ'] [-1.302+0.j]
['ZZZI'] [-2.604+0.j]

---------------------------------------
--- Pauli products for the matrix M2 ---
['IIII'] [5.031+0.j]
['IIZZ'] [-5.031+0.j]
['IZII'] [-1.677+0.j]
['IZZZ'] [1.677+0.j]
['ZIII'] [-3.354+0.j]
['ZIZZ'] [3.354+0.j]

-------------------------------------------------------------------------
--- Circuit of Hamiltonian simulation for the propagator M1 ---
global phase: 3.9062
     ┌────────────┐     ┌───┐┌─────────────┐┌───┐                              »
q_0: ┤ Rz(2.6041) ├─────┤ X ├┤ Rz(-2.6041) ├┤ X ├──────────────────────────────»
     ├────────────┤     └─┬─┘└─────────────┘└─┬─┘          ┌───┐┌─────────────┐»
q_1: ┤ Rz(5.2082) ├───────┼───────────────────┼────────────┤ X ├┤ Rz(-5.2082) ├»
    

In [23]:
# -----------------------------------------------------------------------------------------------
# --- Construct circuit for a diagonal matrix decomposed into tensor products of Pauli matrix ---
# -----------------------------------------------------------------------------------------------
import re
def construct_qucf_circuit_DiagMatrix_PauliDecomposition(A_dec, name_register, n, t, flag_print = True):
    # Here, it is assumed that only I and Z operators present in the Pauli decomposition
    lines_circuit = [None] * len(A_dec) * (2*n-1)
    counter_line = -1
    for one_prod in A_dec:
        if flag_print:
            print()
        pauli_operators = one_prod.paulis[0]
        coef_w = one_prod.coeffs[0]

        if flag_print:
            print("product coefficient: {:21.3e}".format(coef_w))
            print("Pauli product: ", pauli_operators)

        # --- find the target qubits for the product --- 
        positions_of_Z = [match.start() for match in re.finditer('Z', pauli_operators.to_label())]
        if flag_print:
            print("positions of Z in product: ", positions_of_Z)

        if len(positions_of_Z) == 0:
            if flag_print:
                print("skip")
            continue 

        Nz = len(positions_of_Z)
        qs_Z = np.zeros(Nz, dtype=int)
        for ii, pos_Z in enumerate(positions_of_Z):
            qs_Z[ii] = 2*n - 1 - pos_Z
        if flag_print:
            print("target qubits: ", qs_Z)

        # --- the qubit where Rz rotations will be placed ---
        if flag_print:
            print("place Rz at the qubit {:d}".format(qs_Z[-1]))

        # --- construct the circuit for the current product ---
        line_one = ""
        angle_Rz = 2. * t * float(coef_w)
        str_az = "{:0.12e}".format(angle_Rz)
        line_Rz = "gate Rz {:s}[{:d}] {:s} end_gate".format(name_register, qs_Z[-1], str_az)
        if flag_print:
            print("circuit lines:")

        if Nz == 1:
            counter_line += 1
            lines_circuit[counter_line] = line_Rz
            if flag_print:
                print("\t", line_Rz)
        else:
            # --- CNOTs ---
            lines_CNOTS = [None] * (Nz-1)
            for ii in range(Nz-1):
                tq, cq = qs_Z[ii+1], qs_Z[ii]
                line_one = "gate X {:s}[{:d}] control {:s}[{:d}] end_gate".format(
                    name_register, tq, 
                    name_register, cq
                )
                lines_CNOTS[ii] = line_one

            # --- save lines ---
            for line_one_CNOT in lines_CNOTS:
                counter_line += 1
                lines_circuit[counter_line] = line_one_CNOT
                if flag_print:
                    print("\t", line_one_CNOT)

            counter_line += 1
            lines_circuit[counter_line] = line_Rz
            if flag_print:
                print("\t", line_Rz)

            lines_CNOTS.reverse()
            for line_one_CNOT in lines_CNOTS:
                counter_line += 1
                lines_circuit[counter_line] = line_one_CNOT
                if flag_print:
                    print("\t", line_one_CNOT)
        lines_circuit[counter_line] += "\n"
    # --- Adjust the number of elements in the final list ---
    lines_circuit = lines_circuit[:counter_line+1]
    return lines_circuit
# -----------------------------------------------------------------------------------------------
def save_circuit(t, A_dec, name_A, n, name_register, path_save, flag_print_debuggin = True):
    fname = "{:s}_n{:d}_t{:0.2f}.oracle".format(name_A, n, t)
    fullname = path_save + "/" + fname
    print("Storing the circuit into: {:s}".format(fullname))

    # --- constructing circuit ---
    lines_circuit = construct_qucf_circuit_DiagMatrix_PauliDecomposition(
        A_dec, name_register, n, t, flag_print_debuggin
    )

    # --- Write down the circuit into an .oracle file ---
    ff = open(fullname, "w")
    for one_line in lines_circuit:
        ff.write(one_line + "\n")
    ff.close()

    return
# -----------------------------------------------------------------------------------------------
flag_print_debugging = False
save_circuit(t_, M1_dec_, name_M1_, n_, "r", path_save_HS_, flag_print_debugging)
save_circuit(t_, M2_dec_, name_M2_, n_, "r", path_save_HS_, flag_print_debugging)
del flag_print_debugging

# construct_qucf_circuit_DiagMatrix_PauliDecomposition(M1_dec_, "r", 2*n_, t_)

Storing the circuit into: ../QuCF/simulations/Stepanoff/Pauli/hs/M1_n8_t1.00.oracle
Storing the circuit into: ../QuCF/simulations/Stepanoff/Pauli/hs/M2_n8_t1.00.oracle


/tmp/ipykernel_19686/2495365436.py:42: ComplexWarning: Casting complex values to real discards the imaginary part
  angle_Rz = 2. * t * float(coef_w)


In [15]:
# -------------------------------------------------------------
# --- Compare QuCF and theoretical unitaries ---
# -------------------------------------------------------------
import cmath
def compare_diag_U(name_output, diag_U_ref, name_A, flag_print_diags):
    # --- reading QuCF data ---
    dd_loc = qucf_r.read_matrix_sparse(path_qucf_, name_output)
    U_qucf = dd_loc["A"].dense()

    # --- Correcting the global phase ---
    gl_phase = cmath.phase(U_qucf[0,0])
    U_qucf_corr = U_qucf * np.exp(-1j * gl_phase)
    diag_U_qucf = np.sort(mix.get_diag(U_qucf_corr, 0)[0])

    # mix.print_matrix(U_qucf_corr)

    # --- printing diagonals ---
    if flag_print_diags:
        mix.print_array(diag_U_M1_, n_in_row = N_, ff=[21, 3, "e"])
        print()
        mix.print_array(diag_U_qucf, n_in_row = N_, ff=[21, 3, "e"])

    # --- comparison ---
    print("Are the ref and QuCF unitaries close for the propagator {:s}? <<< {:s} >>>".format(
            name_A, str(np.allclose(diag_U_qucf, diag_U_ref, atol = 1e-16))
        )
    )
    return
# -------------------------------------------------------------------------------
flag_print = False
compare_diag_U("hs_OUTPUT.hdf5", diag_U_M1_, "M1", flag_print)
# compare_diag_U("hs_OUTPUT.hdf5", diag_U_M2_, "M2", flag_print)
del flag_print

Reading the matrix from: hs_OUTPUT.hdf5
from the path: ../QuCF/simulations/Stepanoff/Pauli
date of the simulation:  02-06-2025 11:24:33
matrix name:  U
N = 1024
Are the ref and QuCF unitaries close for the propagator M2? <<< True >>>


In [ ]:
# ----------------------------------------------------
# --- Scans ---
# ----------------------------------------------------
def scaling_N_gates_qucf():
    def save_one(t, name_M, x, y, str_y):
        str_t = "t{:0.1f}".format(t)

        x = np.array(x)
        y = np.array(y)
        mix.save_dat_plot_1d_file(
            path_save_scan_ + "/scan_{:s}_{:s}_Nx_{:s}_QuCF.dat".format(name_M, str_t, str_y), 
            x, y
        )
        return
    # ------------------------------------------------------------------
    t = 1.0 

    # ---
    name_M = "M1"
    nx_array        = [  2,  3,   4,   5,    6,    7,    8]
    Ng_array        = [ 15, 57, 185, 545, 1505, 3680, 7605]
    Nx_array = 1 << np.array(nx_array)
    Nx2_array = Nx_array**2
    save_one(t, name_M, Nx_array, Ng_array,  "Ngates")
    save_one(t, name_M, Nx_array, Nx2_array, "Nx2")

    # name_M = "M2"
    # nx_array        = [  2,  3, 4]
    # Ng_array        = [ 15, 57, 185]

    # -----------------------------------------------------------------------
    # --- Approximating the scaling ---
    N_coefs = 3
    def test_func(x, coefs):
        res_scan = coefs[0] * x + coefs[1] * x**(3/2.) + coefs[2] * x**2.
        return res_scan

    coefs = cp.Variable(N_coefs)
    objective = cp.Minimize(cp.sum_squares(
        test_func(Nx_array, coefs) - Ng_array
    ))
    prob = cp.Problem(objective)
    result = prob.solve()
    y_scaling = test_func(Nx_array, coefs.value)
    print("coefs: ", coefs.value)

    fig1 = plt.figure()
    ax = fig1.add_subplot(111)
    ax.loglog(Nx_array, Ng_array, "ob") 
    ax.loglog(Nx_array, y_scaling, "--r") 
    ax.loglog(Nx_array, Nx2_array, "green") 
    plt.xlabel('Nx')
    plt.ylabel("y")
    plt.grid()
    return
# ----------------------------------------------------
scaling_N_gates_qucf()